# Import and Register

Loads models from the import volume and registers them in the target catalog. Idempotent (skips already-imported versions). Restores all aliases that were exported from the source model. Model names in target match source (no suffix).

## Configuration

Read widget parameters: source/target catalogs, schema, volume catalog, and import volume name.

In [ ]:
dbutils.widgets.text("source_catalog", "", "Source catalog")
dbutils.widgets.text("target_catalog", "", "Target catalog")
dbutils.widgets.text("source_schema", "default", "Source schema")
dbutils.widgets.text("target_schema", "default", "Target schema")
dbutils.widgets.text("import_volume", "model_imports", "Import volume name")
dbutils.widgets.text("volume_catalog", "", "Catalog holding import volume (empty = target_catalog)")
dbutils.widgets.text("experiment_prefix", "migration_", "Prefix for the per-model migration experiment ('' = no prefix)")

TARGET_CATALOG = dbutils.widgets.get("target_catalog").strip()
SOURCE_SCHEMA = dbutils.widgets.get("source_schema").strip()
TARGET_SCHEMA = dbutils.widgets.get("target_schema").strip()
IMPORT_VOLUME = dbutils.widgets.get("import_volume").strip()
VOLUME_CATALOG = dbutils.widgets.get("volume_catalog").strip() or TARGET_CATALOG
EXPERIMENT_PREFIX = dbutils.widgets.get("experiment_prefix").strip()
VOLUME_ROOT = f"/Volumes/{VOLUME_CATALOG}/{TARGET_SCHEMA}/{IMPORT_VOLUME}"

if not TARGET_CATALOG:
    raise ValueError("Set target_catalog")
try:
    user_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
except Exception:
    try:
        user_name = spark.conf.get("spark.databricks.userContext.userName")
    except Exception:
        user_name = "unknown"
print(f"Volume: {VOLUME_ROOT}")

## MLflow client and helper functions

Set up the MLflow UC registry client and define helpers for extracting model signatures from metadata, artifacts, or by inference.

In [ ]:
import mlflow, json, os, time, shutil, tempfile, yaml
import numpy as np, pandas as pd
from datetime import datetime
from mlflow import MlflowClient
from mlflow.models import infer_signature, ModelSignature, Model
from mlflow.types.schema import Schema

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

def get_signature_from_metadata(meta):
    sig_dict = meta.get("signature") if isinstance(meta, dict) else None
    if not sig_dict or not isinstance(sig_dict, dict): return None
    inp, out = sig_dict.get("inputs"), sig_dict.get("outputs")
    if not inp: return None
    try:
        return ModelSignature(inputs=Schema.from_dict(inp), outputs=Schema.from_dict(out) if out else None)
    except Exception: return None

def get_signature_from_artifact(artifact_dir):
    if not artifact_dir or not os.path.isdir(artifact_dir): return None
    try:
        mi = mlflow.models.get_model_info(f"file://{os.path.abspath(artifact_dir)}")
        sig = getattr(mi, "signature", None)
        if sig and sig.inputs:
            return ModelSignature(inputs=Schema.from_dict(sig.inputs.to_dict()), outputs=Schema.from_dict(sig.outputs.to_dict()) if sig.outputs else None)
    except Exception: pass
    return None

def infer_signature_from_model(model, params=None, max_features=128):
    if model is None or not hasattr(model, "predict"): return None
    rng = np.random.default_rng(42)
    n = getattr(model, "n_features_in_", None)
    if n is None and params:
        try: n = int(params.get("n_features", 0)) or None
        except (TypeError, ValueError): n = None
        if n is None:
            fn = params.get("feature_names", "") or params.get("feature_names_str", "")
            if isinstance(fn, str) and fn: n = len([x.strip() for x in fn.split(",") if x.strip()])
            elif isinstance(fn, (list, tuple)): n = len(fn)
    if n and n > 0:
        for use_df in (True, False):
            try:
                dummy = pd.DataFrame(rng.normal(0, 1, (2, n)), columns=[f"f{i}" for i in range(n)]) if use_df else np.asarray(rng.normal(0, 1, (2, n)), dtype=np.float64)
                return infer_signature(dummy, model.predict(dummy))
            except Exception: continue
    for n in range(1, max_features + 1):
        try:
            dummy = pd.DataFrame(rng.normal(0, 1, (2, n)), columns=[f"f{i}" for i in range(n)])
            return infer_signature(dummy, model.predict(dummy))
        except Exception: continue
    return None

# ---- Flavor-agnostic helpers (Issue 2 fix) ---------------------------------
# The old import loaded every model with mlflow.sklearn and only fell back to an
# artifact copy by matching the *error message text*, which is fragile. We now
# detect the flavor up front from the MLmodel file and choose the path
# deterministically: sklearn models keep the native sklearn re-log; every other
# flavor (pyfunc, xgboost, lightgbm, pytorch, transformers, ...) is deep-cloned
# as-is and registered, preserving its real flavor.

def read_flavors(model_dir_path):
    try:
        with open(os.path.join(model_dir_path, "MLmodel")) as f:
            return (yaml.safe_load(f) or {}).get("flavors", {}) or {}
    except Exception:
        return {}

def describe_flavor(flavors):
    native = [f for f in flavors if f != "python_function"]
    if native: return native[0]
    if "python_function" in flavors: return "pyfunc"
    return "unknown"

def try_load_native(model_dir_path, flavors):
    label = describe_flavor(flavors)
    for mod_name in ([label] if label not in ("pyfunc", "unknown") else []) + ["pyfunc"]:
        try:
            return getattr(mlflow, mod_name).load_model(model_dir_path)
        except Exception:
            continue
    return None

def materialize_with_signature(model_dir_path, signature):
    """Return a model dir whose MLmodel carries a signature. If the exported
    artifact already has one (or we have none to add), returns the original path;
    otherwise copies to a temp dir and injects the signature so UC registration
    (which requires a signature) succeeds for any flavor."""
    try:
        m = Model.load(model_dir_path)
        if getattr(m, "signature", None) is not None or signature is None:
            return model_dir_path, None
        tmp = tempfile.mkdtemp(prefix="mm_sig_")
        dst = os.path.join(tmp, "model")
        shutil.copytree(model_dir_path, dst)
        m2 = Model.load(dst)
        m2.signature = signature
        m2.save(os.path.join(dst, "MLmodel"))
        return dst, tmp
    except Exception:
        return model_dir_path, None
print(f"MLflow {mlflow.__version__}")


## Read import manifest

Loads the global manifest from the import volume to get the list of models and version counts to import.

In [ ]:
manifest_path = f"{VOLUME_ROOT}/manifest.json"
if not os.path.isfile(manifest_path):
    raise FileNotFoundError(f"Manifest not found: {manifest_path}. Run export & transfer first.")
with open(manifest_path) as f:
    global_manifest = json.load(f)
models_to_import = global_manifest["models"]
print(f"Models to import: {len(models_to_import)}")
for m in models_to_import:
    print(f"  {m['source']} -> {m['target']} ({m['versions']} versions)")

## Import models and restore aliases

For each model in the manifest: loads the sklearn artifact, logs it to a new MLflow run with preserved metrics/params/tags, registers it in the target catalog, and restores every alias that was set on the corresponding source version (no hardcoded list — whatever was exported is applied). Idempotent: skips versions that were already imported (matched by source run ID or source version tag).

In [ ]:
all_model_results = []
for model_entry in models_to_import:
    source_model = model_entry["source"]
    target_model = model_entry["target"]
    model_dir, model_root = model_entry["dir"], f"{VOLUME_ROOT}/{model_entry['dir']}"
    model_short_name = source_model.split(".")[-1]
    experiment_name = f"/Users/{user_name}/{EXPERIMENT_PREFIX}{model_short_name}"
    exp = client.get_experiment_by_name(experiment_name)
    exp_id = exp.experiment_id if exp else client.create_experiment(experiment_name)
    mlflow.set_experiment(experiment_name)
    if not os.path.isdir(model_root):
        raise FileNotFoundError(f"Model directory not found: {model_root}")
    version_dirs = sorted([d for d in os.listdir(model_root) if d.startswith("v") and os.path.isdir(f"{model_root}/{d}")], key=lambda x: int(x[1:]))
    existing_tvs = list(client.search_model_versions(filter_string=f"name='{target_model}'"))
    existing_src_run_ids = set()
    existing_src_keys = set()
    for tv in existing_tvs:
        try:
            tr = client.get_run(tv.run_id)
            tt = dict(tr.data.tags)
            if tt.get("migration.source_run_id"): existing_src_run_ids.add(tt["migration.source_run_id"])
            sm, sv = tt.get("migration.source_model"), tt.get("migration.source_version")
            if sm and sv: existing_src_keys.add((sm, sv))
        except Exception: pass
    results = []
    for vdir_name in version_dirs:
        vnum, vpath = vdir_name[1:], f"{model_root}/{vdir_name}"
        meta_file, model_dir_path = f"{vpath}/metadata.json", f"{vpath}/model"
        if not os.path.exists(meta_file):
            results.append({"v": vnum, "ok": False, "error": "metadata.json missing"}); continue
        with open(meta_file) as f: meta = json.load(f)
        metrics, params, tags = meta.get("metrics", {}), meta.get("params", {}), meta.get("tags", {})
        src_run = meta.get("source_run_id", "")
        if src_run in existing_src_run_ids or (source_model, vnum) in existing_src_keys:
            results.append({"v": vnum, "ok": True, "skipped": True}); continue
        if not os.path.isdir(model_dir_path) or not os.path.isfile(f"{model_dir_path}/MLmodel"):
            if os.path.isfile(f"{vpath}/MLmodel"): model_dir_path = vpath
            else: results.append({"v": vnum, "ok": False, "error": "no MLmodel artifact"}); continue
        tmp_sig = None
        try:
            # Detect the real flavor up front (no reliance on error-message text).
            flavors = read_flavors(model_dir_path)
            flavor_label = describe_flavor(flavors)
            use_sklearn = "sklearn" in flavors
            signature = get_signature_from_metadata(meta) or get_signature_from_artifact(model_dir_path)
            if signature is None:
                signature = infer_signature_from_model(try_load_native(model_dir_path, flavors), params=params)
            input_example = None
            if use_sklearn and params.get("feature_names"):
                fn = [f.strip() for f in params["feature_names"].split(",") if f.strip()]
                input_example = pd.DataFrame(np.random.default_rng(42).normal(0, 1, (1, len(fn))), columns=fn)
            sk_model = None
            if use_sklearn:
                sk_model = mlflow.sklearn.load_model(model_dir_path)
                if signature is None:
                    signature = infer_signature_from_model(sk_model, params=params)
                if signature is None:
                    raise ValueError(f"UC requires a signature; none for {source_model} v{vnum}")
                reg_path = model_dir_path
            else:
                reg_path, tmp_sig = materialize_with_signature(model_dir_path, signature)
                if Model.load(reg_path).signature is None:
                    raise ValueError(f"UC requires a signature; none for {source_model} v{vnum} (flavor={flavor_label})")
            run_name = f"migrated_v{vnum}_{model_short_name}"
            new_version = None
            with mlflow.start_run(experiment_id=exp_id, run_name=run_name) as run:
                for k, v in params.items():
                    try: mlflow.log_param(k, v)
                    except Exception: pass
                for k, v in metrics.items():
                    try: mlflow.log_metric(k, float(v))
                    except Exception: pass
                for k, v in tags.items():
                    if k.startswith("mlflow."): continue
                    try: mlflow.set_tag(k, v)
                    except Exception: pass
                # Run tags allow dotted keys (06_validate lineage checks read these).
                mlflow.set_tag("migration.source_model", source_model)
                mlflow.set_tag("migration.target_model", target_model)
                mlflow.set_tag("migration.source_version", vnum)
                mlflow.set_tag("migration.source_run_id", src_run)
                mlflow.set_tag("migration.type", "volume_deep_clone")
                mlflow.set_tag("migration.flavor", flavor_label)
                _pre = set(v.version for v in client.search_model_versions(f"name='{target_model}'"))
                for attempt in range(3):
                    try:
                        if use_sklearn:
                            mlflow.sklearn.log_model(sk_model=sk_model, name="model", signature=signature, input_example=input_example, registered_model_name=target_model)
                            _post = set(v.version for v in client.search_model_versions(f"name='{target_model}'"))
                            gained = _post - _pre
                            if gained: new_version = sorted(gained, key=lambda x: int(x))[-1]
                        else:
                            # Deep-clone the exported artifact as-is (preserves the flavor), then register.
                            mlflow.log_artifacts(reg_path, artifact_path="model")
                            try: client.create_registered_model(target_model)
                            except Exception: pass
                            mv_reg = mlflow.register_model(f"runs:/{run.info.run_id}/model", target_model)
                            new_version = mv_reg.version
                        break
                    except Exception as e:
                        try:
                            _post = set(v.version for v in client.search_model_versions(f"name='{target_model}'"))
                            gained = _post - _pre
                            if gained:
                                new_version = sorted(gained, key=lambda x: int(x))[-1]; break
                        except Exception: pass
                        if attempt < 2 and ("500" in str(e) or "Retry" in str(e) or "too many" in str(e)):
                            time.sleep(10 * (2 ** attempt))
                        else: raise
            # Defensive: if version-diff detection missed it but registration
            # succeeded, fall back to the latest version so tags still get stamped.
            if new_version is None:
                try:
                    _vs = client.search_model_versions(f"name='{target_model}'")
                    if _vs: new_version = sorted(_vs, key=lambda x: int(x.version))[-1].version
                except Exception: pass
            # Stamp the migration marker on the model VERSION so cleanup can scope
            # reliably without the run. UC version-tag keys cannot contain dots.
            if new_version is not None:
                for k, v in {"migration_source_model": source_model, "migration_target_model": target_model,
                             "migration_source_version": vnum, "migration_source_run_id": src_run,
                             "migration_type": "volume_deep_clone", "migration_flavor": flavor_label}.items():
                    try: client.set_model_version_tag(target_model, new_version, k, v)
                    except Exception: pass
            print(f"  v{vnum} imported (flavor={flavor_label}) -> v{new_version}")
            results.append({"v": vnum, "ok": True, "flavor": flavor_label})
            existing_src_run_ids.add(src_run)
            existing_src_keys.add((source_model, vnum))
        except Exception as e:
            import traceback
            print(f"  v{vnum} FAILED: {type(e).__name__}: {e}")
            print(traceback.format_exc())
            results.append({"v": vnum, "ok": False, "error": f"{type(e).__name__}: {e}"})
        finally:
            if tmp_sig:
                try: shutil.rmtree(tmp_sig)
                except Exception: pass
        time.sleep(1)
    alias_map = {}
    for vd in version_dirs:
        mf = f"{model_root}/{vd}/metadata.json"
        if os.path.exists(mf):
            with open(mf) as f:
                for a in json.load(f).get("aliases", []): alias_map[a] = vd[1:]
    if alias_map:
        all_tv = client.search_model_versions(filter_string=f"name='{target_model}'")
        src_to_tgt = {}
        for tv in all_tv:
            try:
                sv = client.get_run(tv.run_id).data.tags.get("migration.source_version")
                if sv and (sv not in src_to_tgt or int(tv.version) > int(src_to_tgt[sv])): src_to_tgt[sv] = tv.version
            except Exception: pass
        for alias, src_ver in alias_map.items():
            tgt_ver = src_to_tgt.get(src_ver)
            if tgt_ver:
                try: client.set_registered_model_alias(target_model, alias, tgt_ver); print(f"  {alias} -> v{tgt_ver}")
                except Exception as e: print(f"  {alias} FAILED: {e}")
    all_model_results.append({
        "source": source_model,
        "target": target_model,
        "ok": sum(1 for r in results if r["ok"]),
        "fail": sum(1 for r in results if not r["ok"]),
        "experiment": experiment_name,
        "failures": [(r["v"], r.get("error", "unknown")) for r in results if not r["ok"]],
    })

print("="*60)
total_ok, total_fail = 0, 0
failure_lines = []
for r in all_model_results:
    print(f"  {r['source']} -> {r['target']}  ok={r['ok']}  fail={r['fail']}")
    total_ok += r['ok']
    total_fail += r['fail']
    for v, err in r.get("failures", []):
        failure_lines.append(f"{r['target']} v{v}: {err}")
print(f"  TOTAL ok={total_ok}  fail={total_fail}")
print("="*60)
if total_fail > 0 or total_ok == 0:
    detail = "\n".join(failure_lines) if failure_lines else "(no per-version error captured)"
    raise RuntimeError(
        f"Import failed: {total_fail} version(s) failed, {total_ok} succeeded.\n"
        f"Per-version errors:\n{detail}"
    )


## Apply UC permissions (grants) on target models

Reads `grants.json` for each migrated model and re-applies the direct grants on the target registered model via the UC permissions REST endpoint. Each grant is applied independently so a single bad principal (e.g. user/SP that doesn't exist in this metastore) only skips itself rather than failing the whole step. A summary of applied vs skipped grants is printed and any failures cause the step to fail at the end.

In [ ]:
from databricks.sdk import WorkspaceClient
_w = WorkspaceClient()
grants_total_applied = 0
grants_total_skipped = 0
grants_failures = []
for model_entry in models_to_import:
    source_model = model_entry["source"]
    target_model = model_entry["target"]
    model_root = f"{VOLUME_ROOT}/{model_entry['dir']}"
    grants_file = f"{model_root}/grants.json"
    if not os.path.isfile(grants_file):
        print(f"  {target_model}: no grants.json (skip)")
        continue
    with open(grants_file) as f:
        gpayload = json.load(f)
    pas = gpayload.get("privilege_assignments", [])
    if not pas:
        print(f"  {target_model}: 0 grants to apply")
        continue
    applied_here = 0
    print(f"  {target_model}: applying {len(pas)} grant(s)")
    for pa in pas:
        principal = pa.get("principal")
        privs = pa.get("privileges", [])
        if not principal or not privs:
            continue
        try:
            _w.api_client.do(
                "PATCH",
                f"/api/2.1/unity-catalog/permissions/function/{target_model}",
                body={"changes": [{"principal": principal, "add": list(privs)}]},
            )
            print(f"    + {principal}: {privs}")
            applied_here += 1
            grants_total_applied += 1
        except Exception as e:
            msg = str(e)
            if "Could not find principal" in msg or "PRINCIPAL_DOES_NOT_EXIST" in msg or "not found" in msg.lower():
                print(f"    SKIP {principal}: principal not in this metastore")
                grants_total_skipped += 1
            else:
                print(f"    FAIL {principal} ({privs}): {type(e).__name__}: {msg[:200]}")
                grants_failures.append(f"{target_model} {principal} {privs}: {msg[:200]}")
    try:
        verify = _w.api_client.do("GET", f"/api/2.1/unity-catalog/permissions/function/{target_model}")
        actual = verify.get("privilege_assignments", []) if isinstance(verify, dict) else []
        print(f"    => target now has {len(actual)} privilege_assignment(s)")
    except Exception as e:
        print(f"    WARN: could not verify {target_model}: {type(e).__name__}: {e}")
print("="*60)
print(f"GRANTS SUMMARY: applied={grants_total_applied}  skipped={grants_total_skipped}  failed={len(grants_failures)}")
print("="*60)
if grants_failures:
    raise RuntimeError("Grant application had failures:\n" + "\n".join(grants_failures))